# Direct LLM Prompting (Baseline)

This notebook demonstrates how a standard application contacts an LLM directly with a prompt, bypassing any optimization layer. This represents our baseline unoptimized approach.

In [1]:
import requests
import time
import json
import psutil

def query_llm_directly(prompt, model="llama3.2:latest"):
    """
    Direct connection to the LLM (e.g., local Ollama) without dynamic resource profiling.
    """
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    
    print(f"Sending prompt to {model} directly without optimization...")
    start_time = time.time()
    
    cpu_before = psutil.cpu_percent(interval=None)
    ram_before = psutil.virtual_memory().percent
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Error: {e}")
        return None
        
    end_time = time.time()
    latency = end_time - start_time
    cpu_after = psutil.cpu_percent(interval=None)
    
    tokens_generated = data.get('eval_count', 0)
    tokens_per_sec = tokens_generated / latency if latency > 0 else 0
    
    print("\n--- Unoptimized Result ---")
    print(f"Latency: {latency:.2f} seconds")
    print(f"Speed: {tokens_per_sec:.2f} tokens/s")
    print(f"CPU Cost: {cpu_after}%")
    print(f"RAM Cost: {ram_before}%")
    
    # Returning just the first 100 chars to avoid huge outputs
    response_text = data.get('response', '')
    print(f"Response snippet: {response_text[:100]}...")
    return data

In [2]:
prompt = "Explain the concept of quantum entanglement and its implications for faster-than-light communication in exactly 3 paragraphs."
ans = query_llm_directly(prompt, model="llama3.2:latest")

Sending prompt to llama3.2:latest directly without optimization...

--- Unoptimized Result ---
Latency: 30.84 seconds
Speed: 11.64 tokens/s
CPU Cost: 31.5%
RAM Cost: 66.8%
Response snippet: Quantum entanglement is a phenomenon in which two or more particles become connected in such a way t...
